# Notebook 03 — Open-System Rydberg Dynamics (Lindblad)

This notebook adds **open-system effects** to the two-atom Rydberg model using the **Lindblad master equation**.

We study how:
- **spontaneous emission** from the excited / Rydberg-like state
- **pure dephasing**

degrade coherent dynamics and reduce the quality of Rydberg blockade.

The evolution equation is:

$$
\frac{d\rho}{dt} = -i[H, \rho] + \sum_k \left(L_k \rho L_k^\dagger - \frac{1}{2}\{L_k^\dagger L_k, \rho\}\right)
$$

where $L_k$ are collapse operators representing dissipation channels.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, mesolve

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()   # |g><r|

gg = tensor(g, g)
rr = tensor(r, r)

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

n_rr = rr * rr.dag()

## Hamiltonian and collapse operators

In [ ]:
def two_atom_hamiltonian(omega: float, delta: float, V: float):
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def collapse_operators(gamma_decay: float = 0.0, gamma_phi: float = 0.0):
    """Build Lindblad collapse operators.

    gamma_decay: spontaneous emission rate |r> -> |g>
    gamma_phi:   pure dephasing rate on |r>
    """
    c_ops = []

    if gamma_decay > 0:
        c_ops.append(np.sqrt(gamma_decay) * sm1)
        c_ops.append(np.sqrt(gamma_decay) * sm2)

    if gamma_phi > 0:
        c_ops.append(np.sqrt(gamma_phi) * n1)
        c_ops.append(np.sqrt(gamma_phi) * n2)

    return c_ops

## Simulation helpers

In [ ]:
def simulate_open_system(omega: float, delta: float, V: float,
                         gamma_decay: float = 0.0,
                         gamma_phi: float = 0.0,
                         t_final: float = 10.0,
                         n_steps: int = 400):
    """Simulate open-system evolution from |gg>."""
    H = two_atom_hamiltonian(omega, delta, V)
    c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    times = np.linspace(0.0, t_final, n_steps)
    result = mesolve(H, gg, times, c_ops, [n1 + n2, n_rr])
    total_exc = np.real(result.expect[0])
    p_rr = np.real(result.expect[1])
    return times, total_exc, p_rr


def max_double_excitation_open(omega: float, delta: float, V: float,
                               gamma_decay: float = 0.0,
                               gamma_phi: float = 0.0,
                               t_final: float = 10.0,
                               n_steps: int = 400):
    _, _, p_rr = simulate_open_system(
        omega=omega,
        delta=delta,
        V=V,
        gamma_decay=gamma_decay,
        gamma_phi=gamma_phi,
        t_final=t_final,
        n_steps=n_steps,
    )
    return float(np.max(p_rr))

## Time traces: coherent vs dissipative dynamics

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0

cases = [
    (0.0, 0.0, 'coherent'),
    (0.05, 0.0, 'decay only'),
    (0.0, 0.10, 'dephasing only'),
    (0.05, 0.10, 'decay + dephasing'),
]

plt.figure(figsize=(8, 5))
for gamma_decay, gamma_phi, label in cases:
    times, _, p_rr = simulate_open_system(
        omega=omega,
        delta=delta,
        V=V,
        gamma_decay=gamma_decay,
        gamma_phi=gamma_phi,
        t_final=12.0,
        n_steps=500,
    )
    plt.plot(times, p_rr, label=label)

plt.xlabel('Time')
plt.ylabel(r'Double-excitation probability $P_{rr}$')
plt.title('Open-system effects reduce coherent blockade dynamics')
plt.legend()
plt.tight_layout()
plt.show()

## Blockade quality versus decay rate

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.25, 60)
max_p_rr_decay = [
    max_double_excitation_open(
        omega=1.0,
        delta=0.0,
        V=4.0,
        gamma_decay=gd,
        gamma_phi=0.0,
        t_final=12.0,
        n_steps=350,
    )
    for gd in gamma_decay_vals
]

blockade_quality_decay = 1.0 - np.array(max_p_rr_decay)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(gamma_decay_vals, blockade_quality_decay)
plt.xlabel(r'Decay rate $\gamma$')
plt.ylabel(r'Blockade quality $1 - \max(P_{rr})$')
plt.title('Blockade quality versus spontaneous emission')
plt.tight_layout()
plt.show()

## 2D noise landscape over $(\gamma, \gamma_\phi)$

In [ ]:
gamma_decay_grid = np.linspace(0.0, 0.25, 40)
gamma_phi_grid = np.linspace(0.0, 0.25, 40)

quality_landscape = np.zeros((len(gamma_phi_grid), len(gamma_decay_grid)))

for i, gamma_phi in enumerate(gamma_phi_grid):
    for j, gamma_decay in enumerate(gamma_decay_grid):
        max_p_rr = max_double_excitation_open(
            omega=1.0,
            delta=0.0,
            V=4.0,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=10.0,
            n_steps=250,
        )
        quality_landscape[i, j] = 1.0 - max_p_rr

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    quality_landscape,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_grid[0], gamma_decay_grid[-1], gamma_phi_grid[0], gamma_phi_grid[-1]],
)
plt.xlabel(r'Spontaneous emission $\gamma$')
plt.ylabel(r'Pure dephasing $\gamma_\phi$')
plt.title(r'Blockade quality over $(\gamma, \gamma_\phi)$')
plt.colorbar(im, label=r'$1 - \max(P_{rr})$')
plt.tight_layout()
plt.show()

## Interpretation

This notebook adds realism to the blockade model:

- **Spontaneous emission** removes population from the excited state.
- **Dephasing** destroys phase coherence without directly relaxing population.
- Both effects alter the time traces and reduce the quality of coherent control.

The 2D noise landscape provides a compact view of how robust the interaction regime is against these open-system processes.


## Optional next steps

Good follow-ups from here are:

- Bell-state preparation metrics,
- CZ-like gate fidelity,
- distance-dependent interaction $V(R)$,
- pulse-shaping or parameter optimization under noise.
